In [1]:
from pathlib import Path
from glob import glob
import re
import os

import pandas as pd
import numpy as np

In [2]:
from bikipy.border.parallelogram.classes import ParallelogramBorder
from bikipy.border.triangular import TriangularBorder
from bikipy.behaviour.y_maze.experiment import YMaze
from bikipy.readers import DeepLabCutReader

In [3]:
WORKING_DIR = Path("C:/Users/Can/Projects/Neuroscience/bikipy/examples/data")
DATA_DIR = Path("C:/Users/Can/Projects/Neuroscience/Imen/data/y_maze/phd")
EXP_ID_FINDER = re.compile("\d+")

BORDER_IMG_PATH = WORKING_DIR / "images" / "y_maze" / "phd.png"
assert BORDER_IMG_PATH.exists(), f"The image file doesn't exist in {BORDER_IMG_PATH}"
border_img_path_str = str(BORDER_IMG_PATH)

In [4]:
borders = [
        ParallelogramBorder(
            base=[[308.81687801, 193.11825], [288.30224458, 234.14751686]],
            apex=[[174.33205886, 120.17733115], [151.53802172, 158.92719429]],
            guiding_image=border_img_path_str, label="A"
        ),
        ParallelogramBorder(
            base=[[309.95657987, 193.11825], [335.03002072, 234.14751686]],
            apex=[[441.02229344, 116.75822557], [461.53692687, 154.36838686]],
            guiding_image=border_img_path_str, label="B"
        ),
        ParallelogramBorder(
            base=[[290.58164829, 234.14751686], [335.03002072, 234.14751686]],
            apex=[[294.00075387, 394.84547872], [338.44912629, 394.84547872]],
            guiding_image=border_img_path_str, label="C"
        )
    ]

center = TriangularBorder(
    base_a=(289.62987012987014, 235.08441558441552),
    base_b=(332.48701298701303, 235.08441558441552),
    apex=(307.8116883116883, 198.72077922077915),
    label="X"
)


In [5]:
CM_PER_PIXEL = np.linalg.norm(center.base_a - center.base_b) / 5

In [6]:
data_dict = {}
for subdir in os.listdir(str(DATA_DIR)):
    data_dict[subdir] = {}
    for file_path in glob(os.path.join(str(DATA_DIR / subdir), "**.h5")):
        exp_id = EXP_ID_FINDER.findall(Path(file_path).stem)[0]
        print(exp_id)
        dlc_data = DeepLabCutReader.from_hdf(
            file_path, (640, 480), midpoint_groups=(("left_ear", "right_ear"),)
        )
        data_dict[subdir][exp_id] = YMaze(
            dlc_data["mid-left_ear-right_ear"],
            borders,
            center,
            14,
            CM_PER_PIXEL,
            exp_id
        )

10
11
12
13
14
15
16
17
18
19
1
20
21
22
23
24
25
26
27
28
29
2
30
31
32
33
34
35
36
37
38
39
3
40
41
42
43
44
45
46
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
1
20
21
22
23
24
25
26
27
28
29
2
30
31
32
33
34
35
36
37
38
39
3
40
41
42
43
4
5
6
7
8
9


C:\Users\Can\Projects\Neuroscience\bikipy\bikipy\border\base.py:130: UserWarning: Border C has coordinate overlap with other borders
  warn(f"Border {border.label} has coordinate overlap with other borders")


In [7]:
before = YMaze.export_to_dataframe(data_dict["0_before"].values())
after = YMaze.export_to_dataframe(data_dict["1_after"].values())

MultiIndex([(            'Displacement',    ''),
            (              'Mean speed',    ''),
            (       'Mean acceleration',    ''),
            ('Spontaneous alternations',    ''),
            (         'Seconds in area',   'A'),
            (         'Seconds in area',   'B'),
            (         'Seconds in area',   'C'),
            (         'Seconds in area',   'X'),
            (        'Arm alternations',   'A'),
            (        'Arm alternations',   'B'),
            (        'Arm alternations',   'C'),
            (        'Arm alternations',   'X'),
            (     'Triplet alternation', 'ABC'),
            (     'Triplet alternation', 'ACB'),
            (     'Triplet alternation', 'BAC'),
            (     'Triplet alternation', 'BCA'),
            (     'Triplet alternation', 'CAB'),
            (     'Triplet alternation', 'CBA')],
           names=['Feature', 'Area/Triplet'])
MultiIndex([(            'Displacement',    ''),
            (         

In [8]:
with pd.ExcelWriter("C:/Users/Can/Projects/Neuroscience/bikipy/examples/data/phd.xlsx") as writer:
    before.to_excel(writer, sheet_name="Before")
    after.to_excel(writer, sheet_name="After")
